In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import scipy
import sklearn
import math

## Utilities

We define:
1. Neural networks.
2. `fit` function to train the networks.
3. Helper visualization functions.

In [ ]:
from typing import Callable
from torch import nn
from torch.optim import Optimizer
from torch.utils.data import Dataset
from typing import Literal
from tqdm import tqdm


class MLP(nn.Module):
    def __init__(self, dim_in: int, dim_hidden: int, dim_out: int):
        super().__init__()
        self.dim_in = dim_in
        self.dim_hidden = dim_hidden
        self.dim_out = dim_out
        self.net = nn.Sequential(
            nn.Linear(self.dim_in, self.dim_hidden),
            nn.SiLU(),
            nn.Linear(self.dim_hidden, self.dim_hidden),
            nn.SiLU(),
            nn.Linear(self.dim_hidden, self.dim_out),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


class TimedTokenMLP(nn.Module):
    def __init__(self, seq_len: int, dim_hidden: int, vocab_size: int):
        super().__init__()
        self.seq_len = seq_len
        self.dim_hidden = dim_hidden
        self.vocab_size = vocab_size
        flat = dim_hidden * seq_len

        self.embedding = nn.Embedding(vocab_size + 1, dim_hidden)
        self.net = nn.Sequential(
            nn.Linear(flat + 1, flat),
            nn.SiLU(),
        )
        self.out_proj = nn.Linear(dim_hidden, vocab_size)

    def forward(self, inputs: tuple[torch.Tensor, torch.Tensor]) -> torch.Tensor:
        x_t, t = inputs
        h = self.embedding(x_t).flatten(-2, -1)
        h = self.net(torch.cat([h, t], dim=-1))
        h = h.reshape(*h.shape[:-1], self.seq_len, self.dim_hidden)
        return self.out_proj(h)


def fit(
    model: nn.Module,
    *,
    loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    optimizer: Optimizer,
    train_data: Dataset,
    valid_data: Dataset,
    max_n_epochs: int = 100,
    patience: int = 5,
    log_freq: float = 0.1,
):
    train_dataloader = torch.utils.data.DataLoader(
        train_data, batch_size=32, shuffle=True, drop_last=False
    )
    valid_dataloader = torch.utils.data.DataLoader(
        valid_data, batch_size=32, shuffle=False, drop_last=False
    )

    def step(mode: Literal["train", "valid"]):
        dataloader = train_dataloader if mode == "train" else valid_dataloader
        losses: list[float] = []
        with torch.set_grad_enabled(mode == "train"):
            model.train(mode == "train")
            for x, y in dataloader:
                pred = model(x)
                loss = loss_fn(pred, y)
                losses.append(loss.item())

                if mode == "train":
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
        return torch.tensor(losses).mean()

    best_valid = float("inf")
    best_ckpt = model.state_dict()
    cur_patience = patience
    for epoch in tqdm(range(max_n_epochs)):
        train_loss = step("train")
        valid_loss = step("valid")
        if epoch % int(log_freq * max_n_epochs) == 0 or epoch + 1 == max_n_epochs:
            print(f"Epoch: {epoch}, train loss: {train_loss}, valid loss: {valid_loss}")
        if valid_loss < best_valid:
            best_valid = valid_loss
            cur_patience = patience
            best_ckpt = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            cur_patience -= 1
        if cur_patience < 0:
            break

    model.load_state_dict(best_ckpt)
    print("Best ckpt valid loss: ", step("valid"))

In [ ]:
from typing import Callable


def viz_img(
    value_fn: Callable[[torch.Tensor], torch.Tensor], ax, cax=None, cax_label=None
):
    global G_x1_min_display, G_x1_max_display, G_x2_min_display, G_x2_max_display
    GRID_SIZE = 500
    X1, X2 = torch.meshgrid(
        torch.linspace(G_x1_min_display, G_x1_max_display, GRID_SIZE),
        torch.linspace(G_x2_min_display, G_x2_max_display, GRID_SIZE),
        indexing="xy",
    )

    points = torch.stack([X1.ravel(), X2.ravel()], axis=-1)
    v = value_fn(points).reshape(GRID_SIZE, GRID_SIZE)
    im_probability = ax.imshow(
        v,
        extent=[G_x1_min_display, G_x1_max_display, G_x2_min_display, G_x2_max_display],
        origin="lower",
        interpolation="bilinear",
        cmap="Blues",
    )
    plt.colorbar(im_probability, cax=cax, label=cax_label)


def viz_scatter(
    samples: torch.Tensor,
    value_fn: Callable[[torch.Tensor], torch.Tensor],
    ax,
    cax=None,
    cax_label=None,
):
    v = value_fn(samples)
    im = ax.scatter(
        samples[:, 0],
        samples[:, 1],
        s=5,
        c=v,
        alpha=1.0,
    )
    plt.colorbar(im, cax=cax, label=cax_label)

# Problem Setup
In this notebook, we operate in a synthetic 2D continuous space. Instead of working with complex image pixels, this low-dimensional setup allows us to easily visualize how diffusion models manipulate probability densities, vector fields, and sampling trajectories.

----

We define a custom target distribution $p_X(x)$ called ThickSineDensity.

The Geometry: The mass of the distribution is concentrated along a sinusoidal curve $x_2 = \sin(\omega \pi x_1)$ bounded within a specific horizontal window $[x_1^{\text{min}}, x_1^{\text{max}}]$.

The Noise/Thickness: The distribution has a uniform thickness defined by $\sigma$. Points are distributed vertically around the sine wave according to a cosine-shaped probability density function, dropping cleanly to zero outside the boundary ($r > 1.0$).

----

Our goal is to treat this distribution as the unknown "true data distribution". The generative model will only have access to a finite set of empirical samples generated from this distribution, mimicking real-world scenarios.

## Target Distribution

### Distribution definition

In [ ]:
from torch.distributions import Distribution, constraints


class ThickSineDensity(Distribution):
    arg_constraints = {
        "omega": constraints.positive,
        "sigma": constraints.positive,
        "x1_min": constraints.real,
        "x1_max": constraints.real,
    }
    has_rsample = False

    def __init__(
        self,
        omega: float = 2.0,
        sigma: float = 0.08,
        x1_min: float = -2.0,
        x1_max: float = 2.0,
    ):
        self.omega = torch.as_tensor(omega, dtype=torch.float32)
        self.sigma = torch.as_tensor(sigma, dtype=torch.float32)
        self.x1_min = torch.as_tensor(x1_min, dtype=torch.float32)
        self.x1_max = torch.as_tensor(x1_max, dtype=torch.float32)
        super().__init__(
            batch_shape=torch.Size([]),
            event_shape=torch.Size([2]),
            validate_args=True,
        )

    def _mean_curve(self, t: torch.Tensor) -> torch.Tensor:
        return torch.sin(self.omega * math.pi * t)

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., 0]
        x2 = x[..., 1]

        mu = self._mean_curve(x1)
        r = (x2 - mu).abs() / self.sigma

        inside = (x1 >= self.x1_min) & (x1 <= self.x1_max) & (r <= 1.0)

        w = torch.cos((math.pi / 2) * r.clamp(max=1.0))
        log_normalizer = torch.log(
            (self.x1_max - self.x1_min) * (4.0 * self.sigma / math.pi)
        )

        return torch.where(
            inside,
            torch.log(w) - log_normalizer,
            torch.full_like(w, float("-inf")),
        )

    def sample(self, sample_shape: torch.Size = torch.Size()) -> torch.Tensor:
        shape = self._extended_shape(sample_shape)
        n = shape[:-1].numel()

        with torch.no_grad():
            x1 = torch.empty(n).uniform_(self.x1_min.item(), self.x1_max.item())
            x2 = torch.empty(n)
            mu = self._mean_curve(x1)

            pending = torch.ones(n, dtype=torch.bool)
            while pending.any():
                n_pending = int(pending.sum())

                delta = torch.empty(n_pending).uniform_(
                    -self.sigma.item(), self.sigma.item()
                )
                accept_prob = torch.cos((math.pi / 2) * delta.abs() / self.sigma)
                accepted = torch.rand(n_pending) < accept_prob

                if accepted.any():
                    idx = pending.nonzero(as_tuple=False).squeeze(1)[accepted]
                    x2[idx] = mu[idx] + delta[accepted]
                    pending[idx] = False

        return torch.stack([x1, x2], dim=-1).reshape(shape)

    def viz_support(self, ax):
        t = torch.linspace(self.x1_min, self.x1_max, 4000)
        curve = self._mean_curve(t)
        curve_lower = curve - self.sigma
        curve_upper = curve + self.sigma
        ax.plot(t, curve_upper, color="navy", linestyle="--", linewidth=2)
        ax.plot(t, curve_lower, color="navy", linestyle="--", linewidth=2)

### Global variables definition

In [ ]:
omega = 1.0
sigma = 1.0
x1_min = -3.0
x1_max = 2.0
x2_min = -1.0 - sigma
x2_max = 1.0 + sigma
N_gen = 400
N_clf = 50

G_x1_min_display = x1_min - 1
G_x1_max_display = x1_max + 1
G_x2_min_display = x2_min - 1
G_x2_max_display = x2_max + 1

G_gt_distribution = ThickSineDensity(
    omega=omega, sigma=sigma, x1_min=x1_min, x1_max=x1_max
)

### Visualization

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.0, 1.0]})

ax_probability = axs[0]
ax_samples = axs[1]

ax_samples.set_title("Samples")
ax_probability.set_title("Probability")

viz_img(
    lambda x: G_gt_distribution.log_prob(x).exp(), ax_probability, cax_label="density"
)

samples = G_gt_distribution.sample((N_gen,))
ax_samples.scatter(samples[:, 0], samples[:, 1], alpha=0.2)

for ax in [ax_samples, ax_probability]:
    G_gt_distribution.viz_support(ax)
    ax.set_xlim(G_x1_min_display, G_x1_max_display)
    ax.set_ylim(G_x2_min_display, G_x2_max_display)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_aspect("equal")


plt.tight_layout()
plt.show()

## Objective-Driven Generation via Reward Functions

In standard generative modeling, our only goal is to mimic the training data distribution $p_X(x)$. However, in many real-world tasks (e.g., drug discovery, material design, or reinforcement learning alignment), we don't just want any valid sample. We want high-utility samples.

The Challenge: We want to steer our generator to sample from regions where both the ground-truth data density $p_X(x)$ is high (so the sample is "realistic") and the reward $R(x)$ is maximized.

### Example reward definition

In [ ]:
G_reward_fn = lambda x: x[..., -1] ** 2

### Visualization

In [ ]:
samples = G_gt_distribution.sample((1000,))
viz_scatter(samples, G_reward_fn, plt.gca(), cax_label="rewards")
G_gt_distribution.viz_support(plt.gca())

## Metrics

To measure the performance of the generator and guidance we introduce several empirical measures.

### Definitions

__Reward optimization metrics__

- **`rewards_mean`** — Average reward across all generated samples, with invalid samples contributing zero.
  - *Strengths*: Single summary number for overall optimization quality; easy to track across runs.
  - *Limitations*: In optimized generation we usually care about the model's ability to produce very high-reward samples, not just generally okay ones.

- **`rewards_max`** — The single best reward value found among the generated samples.
  - *Strengths*: Directly answers "what's the best thing this model can produce"; useful for best-case/exploration-oriented evaluation.
  - *Limitations*: High variance — a single lucky (or unlucky) sample dominates; not representative of typical behavior; can be gamed by generating more samples.

- **`rewards_q95`** — Reward value at the 95th percentile, capturing the quality of the best-performing tail of samples.
  - *Strengths*: Useful when the goal is to find a few very good samples (e.g., drug discovery, design search) rather than optimize the average case; more robust than the mean to the bulk of low/zero-reward samples.
  - *Limitations*: Ignores most of the distribution.

---
__Feasibility metrics__

- **`MMD^2`** — Squared Maximum Mean Discrepancy between generated samples and a reference dataset, capturing distributional similarity.
  - *Strengths*: Principled kernel-based two-sample statistic capturing differences in location and shape, not just first/second moments; lower values mean closer match to the reference.
  - *Limitations*: Sensitive to kernel bandwidth (`gamma`, fixed at 1.0 here, may not suit all data scales); quadratic cost in sample size; hard to interpret in absolute terms (relative/comparative, not an intuitive distance).

- **`validity_mean`** — Fraction of generated samples that fall within the support of the reference distribution (i.e., have nonzero probability).
  - *Strengths*: Simple, interpretable as a percentage; measures feasibility/constraint satisfaction independent of reward magnitude.
  - *Limitations*: Only practical when an oracle is available to evaluate whether a sample is valid.
---
__Diversity metrics__

- **`pairwisedistance_mean`** — Average pairwise Euclidean distance between generated samples, indicating how spread out they are.
  - *Strengths*: Simple, intuitive measure of intra-batch diversity/mode coverage; helps detect mode collapse (collapsed samples cluster together, giving a low mean distance).
  - *Limitations*: Ignores the reference distribution, so high diversity isn't necessarily "good" if it covers implausible/low-density regions; quadratic cost; sensitive to input scale, so not directly comparable across domains.

In [ ]:
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import rbf_kernel
from dataclasses import dataclass, field
from typing import Callable


@dataclass
class SampledDistributionToOptimize:
    unlabeled_x: torch.Tensor
    labeled_x: torch.Tensor
    labeled_y: torch.Tensor
    compute_metrics: Callable[[torch.Tensor], dict[str, float]] = field(repr=False)


class DistributionToOptimize:
    def __init__(
        self,
        distribution: Distribution,
        reward_fn: Callable[[torch.Tensor], torch.Tensor],
    ):
        self.distribution = distribution
        self.reward_fn = reward_fn

    def sample_train_data(
        self, N_gen: int, N_clf: int
    ) -> SampledDistributionToOptimize:
        train_x_gen = self.distribution.sample((N_gen,))

        def compute_metrics(samples: torch.Tensor):
            assert samples.shape[-1] == 2
            samples = samples.reshape(-1, samples.shape[-1])

            probs = self.distribution.log_prob(samples).exp()
            rewards = self.reward_fn(samples) * (probs > 0)
            pairwise_dists = pdist(samples)
            mmd_sq = (
                rbf_kernel(samples, samples, gamma=1.0).mean()
                + rbf_kernel(train_x_gen, train_x_gen, gamma=1.0).mean()
                - 2 * rbf_kernel(samples, train_x_gen, gamma=1.0).mean()
            )

            return {
                "rewards_mean": rewards.mean().item(),
                "rewards_max": rewards.max().item(),
                "rewards_q95": torch.quantile(rewards, 0.95).item(),
                "MMD^2": mmd_sq.item(),
                "validity_mean": (probs > 0).float().mean().item(),
                "pairwisedistance_mean": pairwise_dists.mean().item(),
            }

        return SampledDistributionToOptimize(
            unlabeled_x=train_x_gen,
            labeled_x=train_x_gen[:N_clf],
            labeled_y=self.reward_fn(train_x_gen[:N_clf]),
            compute_metrics=compute_metrics,
        )

### Global variables definition

In [ ]:
G_distribution_to_optimize = DistributionToOptimize(G_gt_distribution, G_reward_fn)
G_sampled_distribution_to_optimize = G_distribution_to_optimize.sample_train_data(
    N_gen, N_clf
)

### Example metrics computation

In [ ]:
G_sampled_distribution_to_optimize.compute_metrics(
    G_sampled_distribution_to_optimize.unlabeled_x
)

### Training reward surogate

In [ ]:
G_reward_net = MLP(2, 32, 1)
from sklearn.model_selection import train_test_split

train_x, valid_x, train_y, valid_y = train_test_split(
    G_sampled_distribution_to_optimize.labeled_x,
    G_sampled_distribution_to_optimize.labeled_y,
    test_size=0.2,
)

train_data = torch.utils.data.TensorDataset(train_x.float(), train_y.float())
valid_data = torch.utils.data.TensorDataset(valid_x.float(), valid_y.float())

fit(
    G_reward_net,
    loss_fn=lambda x, y: torch.nn.functional.mse_loss(x.flatten(), y.flatten()),
    optimizer=torch.optim.SGD(
        G_reward_net.parameters(), lr=0.001
    ),  # TODO: maybe better optimizer would help?
    train_data=train_data,
    valid_data=valid_data,
    max_n_epochs=1000,
    patience=20,
)
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].set_title("Reward ground-truth")
axs[1].set_title("Reward prediction")

viz_img(G_reward_fn, axs[0])
with torch.no_grad():
    viz_img(G_reward_net, axs[1])

# Discrete Diffusion

## 1. Setup: The Mask Token

Let $x_0 \in \mathcal{V}^d$ be a sequence of $d$ tokens over vocabulary $\mathcal{V}$. We introduce a special symbol $[M] \notin \mathcal{V}$ called the **mask token**, and let $\alpha_t \in [0,1]$ be a noise schedule with $\alpha_0 = 1$ (clean) and $\alpha_1 = 0$ (fully corrupted).

---

## 2. Marginal Corruption Distribution

At time $t$, each token independently survives with probability $\alpha_t$ or is replaced by $[M]$ with probability $1-\alpha_t$:

$$
q_t(x_t \mid x_0) = \prod_{i=1}^{d}
\Bigl[
  \alpha_t\,\mathbf{1}_{x_t^{(i)} = x_0^{(i)}}
  +
  (1-\alpha_t)\,\mathbf{1}_{x_t^{(i)} = [M]}
\Bigr].
$$

---

## 3. Factorized Denoising Model

The true posterior $p(x_0 \mid x_t)$ is intractable — it requires reasoning over all $|\mathcal{V}|^d$ possible sequences jointly. Instead, we learn a **factorized** approximation:

$$p_\theta(x_0 \mid x_t, t) = \prod_{i=1}^d p_\theta\!\left(x_0^{(i)} \mid x_t, t\right),$$

trained with cross-entropy loss:

$$\mathcal{L}(\theta) = \mathbb{E}_{x_0,\, x_t \sim q_t}\!\left[-\sum_{i=1}^d \log p_\theta\!\left(x_0^{(i)} \mid x_t, t\right)\right].$$

The optimal solution for each position $i$ is the **marginal** posterior:

$$p_\theta^*\!\left(x_0^{(i)} = k \mid x_t, t\right) = p\!\left(x_0^{(i)} = k \mid x_t, t\right).$$

This product of marginals is not the full joint — it ignores inter-position correlations:

$$\prod_{i} p\!\left(x_0^{(i)} \mid x_t, t\right) \;\neq\; p(x_0 \mid x_t, t) \quad \text{in general.}$$

One might worry that learning only these marginals is insufficient to drive the generative process toward the correct distribution. Section 6 argues that in the continuous-time limit this worry is unfounded: the per-position posteriors are exactly what is needed to specify the reverse process, with no information lost.

**Note on the training objective.** The cross-entropy loss above is the natural objective for absorbing masking, where the ELBO reduces exactly to predicting the clean token at masked positions. For uniform noising the ELBO does not simplify this way; the appropriate objective is instead a KL divergence between the true and model reverse posteriors at each step, or equivalently the **score entropy** loss that directly targets the ratios $p_t(x_t^{(i)} = v)/p_t(x_t^{(i)} = u)$ entering the reverse rates. See §5b.

---

## 4. Forward Process

Once a token is masked, it **stays masked**. The transition from $s < t$ to $t$ is:

$$
q\!\left(x_t^{(i)} \mid x_s^{(i)}, x_0^{(i)}\right) =
\begin{cases}
1 & x_s^{(i)} = [M],\ x_t^{(i)} = [M] \\[4pt]
\dfrac{\alpha_t}{\alpha_s} & x_s^{(i)} = x_0^{(i)},\ x_t^{(i)} = x_0^{(i)} \\[6pt]
1 - \dfrac{\alpha_t}{\alpha_s} & x_s^{(i)} = x_0^{(i)},\ x_t^{(i)} = [M]
\end{cases}
$$

Masked tokens are **never re-revealed** in the forward process, making each position's mask state monotone in $t$. Note that the transition kernel depends on $x_0^{(i)}$: one must know the clean token to distinguish the unmasked state from the masked state. As $t - s \to 0$, the probability of any transition vanishes ($\alpha_t/\alpha_s \to 1$), so $x_t$ and $x_{t+dt}$ are strongly coupled — they agree with high probability and differ only at an infinitesimal rate.

---

## 5. Reverse (Denoising) Step

Generation proceeds $x_1 \to x_{t_{N-1}} \to \cdots \to x_0$. At each step $t \to s$ we need $q(x_s^{(i)} \mid x_t^{(i)}, x_0^{(i)})$, which we approximate by plugging in $p_\theta(x_0^{(i)} \mid x_t, t)$.

Unmasked positions carry forward unchanged: $x_s^{(i)} = x_t^{(i)}$ if $x_t^{(i)} \neq [M]$. For masked positions, the reverse posterior follows from Bayes' theorem on the absorbing kernel:

$$
q\!\left(x_s^{(i)} \mid x_t^{(i)}{=}[M],\, x_0^{(i)}\right) =
\begin{cases}
\dfrac{\alpha_s - \alpha_t}{1 - \alpha_t} & x_s^{(i)} = x_0^{(i)} \\[6pt]
\dfrac{1 - \alpha_s}{1 - \alpha_t} & x_s^{(i)} = [M]
\end{cases}
$$

Substituting the model for the unknown $x_0^{(i)}$:

$$
x_s^{(i)} =
\begin{cases}
k \sim p_\theta\!\left(x_0^{(i)} {=} k \mid x_t, t\right)
  & \text{w.p. } \dfrac{\alpha_s - \alpha_t}{1 - \alpha_t}, \\[6pt]
[M]
  & \text{w.p. } \dfrac{1 - \alpha_s}{1 - \alpha_t}.
\end{cases}
\qquad x_t^{(i)} = [M].
$$



## (Extra) Uniform Noising
An alternative to masking process is uniform noise process.

### Forward Process

At each time $t$, each token independently either survives or is replaced by a token drawn uniformly from $\mathcal{V}$:

$$q_t\!\left(x_t^{(i)} \mid x_0^{(i)}\right) = \alpha_t\,\mathbf{1}_{x_t^{(i)} = x_0^{(i)}} + \frac{1-\alpha_t}{|\mathcal{V}|}.$$

$$q\!\left(x_t^{(i)} \mid x_s^{(i)}\right) = \frac{\alpha_t}{\alpha_s}\,\mathbf{1}_{x_t^{(i)} = x_s^{(i)}} + \left(1 - \frac{\alpha_t}{\alpha_s}\right)\frac{1}{|\mathcal{V}|}.$$

### Reverse (Denoising) Step
This case is different than absorbing masking process. The reverse continuous-time Markov chain is not determined by the clean-token posteriors $p(x_0^{(i)}\mid x_t)$. Instead, the sufficient statistic is the **discrete score**, namely probability ratios of the form
$$\frac{p_t(x_t^{(i)}=v)}{p_t(x_t^{(i)}=u)},$$
which directly determine the reverse jump rates. Consequently, modern continuous-time formulations such as SEDD do not learn $p(x_0^{(i)}\mid x_t)$. Rather, they learn these score ratios directly via the score-entropy objective and construct the reverse process from them. Thus, while absorbing diffusion admits an $x_0$-prediction parameterization whose learned marginals are sufficient in the continuous-time limit, uniform diffusion is more naturally parameterized by the discrete score itself. You can read more in Lou et al. (SEDD, 2024).

## Target Distribution

In [ ]:
class DiscretizedDistribution(Distribution):
    arg_constraints = {}
    has_rsample = False

    def __init__(
        self,
        distribution: Distribution,
        n_bins: int,
        n_mc_samples: int = 200_000,
    ):
        assert len(distribution.event_shape) == 1, (
            "DiscretizedDistribution requires a flat event shape, "
            f"got {distribution.event_shape}"
        )
        super().__init__(
            batch_shape=distribution.batch_shape,
            event_shape=distribution.event_shape,
        )
        self.base_distribution = distribution
        self.n_bins = n_bins
        self.n_dims = distribution.event_shape[0]

        with torch.no_grad():
            mc = distribution.sample((n_mc_samples,))

        self.boundaries = []
        for d in range(self.n_dims):
            vals = mc[:, d]
            lo, hi = vals.min().item(), vals.max().item()
            bounds_d = torch.linspace(lo, hi, n_bins - 1)
            self.boundaries.append(bounds_d)

        dim_indices = self._quantize(mc)
        strides = torch.tensor(
            [n_bins ** (self.n_dims - 1 - d) for d in range(self.n_dims)]
        )
        flat = (dim_indices * strides).sum(dim=-1)

        counts = torch.bincount(flat, minlength=n_bins**self.n_dims).float()
        joint = counts.reshape((n_bins,) * self.n_dims)

        joint = joint + 1e-6
        joint = joint / joint.sum()

        self.log_joint = joint.log()

    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        idx = tuple(x[..., d].long() for d in range(self.n_dims))
        return self.log_joint[idx]

    def sample(self, sample_shape: torch.Size = torch.Size()) -> torch.Tensor:
        with torch.no_grad():
            continuous = self.base_distribution.sample(sample_shape)
        return self._quantize(continuous)

    def _quantize(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stack(
            [
                torch.bucketize(x[..., d].contiguous(), self.boundaries[d])
                for d in range(self.n_dims)
            ],
            dim=-1,
        )

    def to_continuous(self, x: torch.Tensor) -> torch.Tensor:
        result = []
        for d in range(self.n_dims):
            bounds = self.boundaries[d]
            step = bounds[1] - bounds[0]
            aug = torch.cat([bounds[:1] - step, bounds, bounds[-1:] + step])
            lo = aug[x[..., d].long()]
            hi = aug[x[..., d].long() + 1]
            result.append((lo + hi) / 2)

        return torch.stack(result, dim=-1)

    def log_prob_continuous(self, x: torch.Tensor) -> torch.Tensor:
        return self.log_prob(self._quantize(x))

    def viz_support(self, ax):
        if hasattr(self.base_distribution, "viz_support"):
            self.base_distribution.viz_support(ax)

In [ ]:
G_discretized_gt_distribution = DiscretizedDistribution(G_gt_distribution, 20)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

ax_cont = axs[0]
ax_disc = axs[1]

viz_img(lambda x: G_gt_distribution.log_prob(x).exp(), ax_cont, cax_label="density")
G_gt_distribution.viz_support(ax_cont)
ax_cont.set_title("Continuous $p_X(x)$")
ax_cont.set_xlabel(r"$x_1$")
ax_cont.set_ylabel(r"$x_2$")

viz_img(
    lambda x: G_discretized_gt_distribution.log_prob_continuous(x).exp(),
    ax_disc,
    cax_label="probability",
)
G_discretized_gt_distribution.viz_support(ax_disc)
ax_disc.set_title(
    f"Discretized $p_X(x)$  ({G_discretized_gt_distribution.n_bins} bins/dim)"
)
ax_disc.set_xlabel(r"$x_1$")
ax_disc.set_ylabel(r"$x_2$")

for ax in axs:
    ax.set_xlim(G_x1_min_display, G_x1_max_display)
    ax.set_ylim(G_x2_min_display, G_x2_max_display)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Mask-Absorbed Noising

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def viz_masked_noised_joint(
    discretized_dist: DiscretizedDistribution, alpha_t: float
) -> np.ndarray:
    joint = discretized_dist.log_joint.exp()
    n = discretized_dist.n_bins
    p1 = joint.sum(dim=1)
    p2 = joint.sum(dim=0)

    mat = np.zeros((n + 1, n + 1), dtype=np.float32)
    mat[1 : n + 1, 0] = (alpha_t * (1 - alpha_t) * p2).numpy()
    mat[0, 1 : n + 1] = ((1 - alpha_t) * alpha_t * p1).numpy()

    mat[1 : n + 1, 1 : n + 1] = (alpha_t**2 * joint.T).numpy()
    mat[0, 0] = (1 - alpha_t) ** 2
    return mat


alpha_values = [1.0, 0.95, 0.9, 0.85, 0.8]
n = G_discretized_gt_distribution.n_bins
N = len(alpha_values)
sep = 0.5

cmap_main = plt.cm.Blues
cmap_mask = plt.cm.Oranges

figsize = (4.0 * N, 4.8)

fig, axs = plt.subplots(1, N, figsize=figsize, squeeze=False)
axs = axs[0]

tick_pos = list(range(0, n, 5)) + [n]
tick_lab = [str(i) for i in range(0, n, 5)] + ["[M]"]

for ax, alpha_t in zip(axs, alpha_values):
    mat = viz_masked_noised_joint(G_discretized_gt_distribution, alpha_t)
    norm = mcolors.Normalize(vmin=0, vmax=mat.max())

    rgba = np.zeros((n + 1, n + 1, 4), dtype=np.float32)
    rgba[1 : n + 1, 1 : n + 1] = cmap_main(norm(mat[1 : n + 1, 1 : n + 1]))
    rgba[1 : n + 1, 0] = cmap_mask(norm(mat[1 : n + 1, 0]))
    rgba[0, :] = cmap_mask(norm(mat[0, :]))

    ax.imshow(rgba, origin="lower", aspect="equal", interpolation="nearest")

    # separator lines
    ax.axhline(sep, color="black", linewidth=1.5, alpha=0.7)
    ax.axvline(sep, color="black", linewidth=1.5, alpha=0.7)

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_lab)
    ax.set_yticks(tick_pos)
    ax.set_yticklabels(tick_lab)
    ax.set_title(rf"$\alpha_t = {alpha_t:.2f}$", fontsize=12)
    ax.set_xlabel(r"$x_t^{(1)}$")

axs[0].set_ylabel(r"$x_t^{(2)}$")

# --- legend ---
from matplotlib.patches import Patch

fig.legend(
    handles=[
        Patch(facecolor=cmap_main(0.65), label="visible token probability (Blues)"),
        Patch(facecolor=cmap_mask(0.65), label="[M] token probability (Oranges)"),
    ],
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, -0.06),
    fontsize=10,
    frameon=False,
)

fig.suptitle(
    r"Forward masking: joint distribution $P(x_t^{(1)},\, x_t^{(2)})$",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np


def viz_uniform_noised_joint(
    discretized_dist: DiscretizedDistribution, alpha_t: float
) -> np.ndarray:
    joint = discretized_dist.log_joint.exp().numpy().T
    n = discretized_dist.n_bins

    p1 = joint.sum(axis=1, keepdims=True)
    p2 = joint.sum(axis=0, keepdims=True)
    both_survived = (alpha_t**2) * joint

    dim1_survived = alpha_t * (1 - alpha_t) * p1 * (1.0 / n)
    dim2_survived = (1 - alpha_t) * alpha_t * (1.0 / n) * p2
    both_noised = ((1 - alpha_t) ** 2) * (1.0 / (n**2))
    mat = both_survived + dim1_survived + dim2_survived + both_noised
    return mat


alpha_values = [1.0, 0.8, 0.5, 0.2, 0.1]
n = G_discretized_gt_distribution.n_bins
N = len(alpha_values)

cmap = plt.cm.Blues
figsize = (4.0 * N, 4.5)

fig, axs = plt.subplots(1, N, figsize=figsize, squeeze=False)
axs = axs[0]

tick_pos = list(range(0, n, 5))
tick_lab = [str(i) for i in range(0, n, 5)]

max_val = G_discretized_gt_distribution.log_joint.exp().max().item()
norm = mcolors.Normalize(vmin=0, vmax=max_val)

for ax, alpha_t in zip(axs, alpha_values):
    mat = viz_uniform_noised_joint(G_discretized_gt_distribution, alpha_t)
    im = ax.imshow(
        mat,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
        cmap=cmap,
        norm=norm,
    )

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_lab)
    ax.set_yticks(tick_pos)
    ax.set_yticklabels(tick_lab)
    ax.set_title(rf"$\alpha_t = {alpha_t:.2f}$", fontsize=12)
    ax.set_xlabel(r"$x_t^{(1)}$")

axs[0].set_ylabel(r"$x_t^{(2)}$")


fig.subplots_adjust(right=0.85)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.7])
fig.colorbar(im, cax=cbar_ax, label="Probability Mass")

fig.suptitle(
    r"Forward Uniform Noising: joint distribution $P(x_t^{(1)},\, x_t^{(2)})$",
    fontsize=13,
    y=0.98,
)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()

## Denoising


In [ ]:
import torch.nn.functional as F

MASK_TOKEN: int = G_discretized_gt_distribution.n_bins


class MaskingDataset:
    def __init__(
        self,
        data: torch.Tensor,
        *,
        alpha_schedule: Callable[[torch.Tensor], torch.Tensor],
        mask_token: int,
        t_batch_size: int = 2,
        mask_batch_size: int = 16,
    ):
        self.data = data
        self.alpha_schedule = alpha_schedule
        self.mask_token = mask_token
        self.t_batch_size = t_batch_size
        self.mask_batch_size = mask_batch_size

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        x0 = self.data[idx]
        L = x0.shape[0]

        t = torch.rand(self.t_batch_size, 1, 1).expand(
            self.t_batch_size, self.mask_batch_size, 1
        )

        alpha_t = self.alpha_schedule(t)

        x0_exp = x0[None, None, :].expand(self.t_batch_size, self.mask_batch_size, L)

        keep = torch.bernoulli(alpha_t.expand(-1, -1, L)).bool()

        x_t = torch.where(keep, x0_exp, x0_exp.new_full(x0_exp.shape, self.mask_token))
        return (x_t, t), (x0_exp, ~keep)

In [ ]:
from sklearn.model_selection import train_test_split

G_alpha_schedule = lambda t: 1.0 - t

G_discrete_denoiser = TimedTokenMLP(
    seq_len=2,
    dim_hidden=64,
    vocab_size=G_discretized_gt_distribution.n_bins,
)

train_x_disc, valid_x_disc = train_test_split(
    G_discretized_gt_distribution._quantize(
        G_sampled_distribution_to_optimize.unlabeled_x
    ),
    test_size=0.2,
)

train_data_disc = MaskingDataset(
    train_x_disc,
    alpha_schedule=G_alpha_schedule,
    mask_token=MASK_TOKEN,
)
valid_data_disc = MaskingDataset(
    valid_x_disc,
    alpha_schedule=G_alpha_schedule,
    mask_token=MASK_TOKEN,
)


def masked_cross_entropy_loss(
    pred: torch.Tensor, target: tuple[torch.Tensor, torch.Tensor]
):
    x0, mask_idx = target
    if not mask_idx.any():
        return torch.tensor(0.0, requires_grad=True)
    return F.cross_entropy(pred[mask_idx], x0[mask_idx])


fit(
    G_discrete_denoiser,
    loss_fn=masked_cross_entropy_loss,
    optimizer=torch.optim.AdamW(G_discrete_denoiser.parameters(), lr=1e-2),
    train_data=train_data_disc,
    valid_data=valid_data_disc,
    max_n_epochs=1000,
    patience=100,
)

## Sampling

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
from typing import Callable


def pred_x0_disc(
    discrete_denoiser: nn.Module, xt: torch.Tensor, t: float
) -> torch.Tensor:
    t_tensor = torch.full((*xt.shape[:-1], 1), t, dtype=torch.float32, device=xt.device)
    return discrete_denoiser((xt, t_tensor))


@torch.no_grad()
def sample_discrete_diffusion(
    discrete_denoiser: nn.Module,
    x1: torch.Tensor,
    alpha_schedule: Callable[[torch.Tensor], torch.Tensor],
    num_steps: int = 100,
) -> torch.Tensor:
    mask_token = discrete_denoiser.vocab_size
    xt = x1.clone()

    ts = torch.linspace(1.0, 0.0, num_steps + 1, device=xt.device)
    alphas = alpha_schedule(ts)

    for i in range(num_steps):
        t_curr = ts[i].item()
        alpha_curr = alphas[i].item()
        alpha_next = alphas[i + 1].item()
        is_masked = xt == mask_token

        if not is_masked.any():
            continue

        logits = pred_x0_disc(discrete_denoiser, xt, t_curr)

        probs = torch.softmax(logits, dim=-1)
        x0_pred = Categorical(probs).sample()

        p_unmask = (alpha_next - alpha_curr) / (1.0 - alpha_curr + 1e-9)
        p_unmask = max(0.0, min(1.0, p_unmask))

        unmask_trigger = torch.bernoulli(
            torch.full_like(xt, p_unmask, dtype=torch.float32)
        ).bool()

        should_reveal = is_masked & unmask_trigger
        xt = torch.where(should_reveal, x0_pred, xt)

    return xt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

N_SAMPLES = 1000
G_discrete_denoiser.eval()
with torch.no_grad():
    samples = sample_discrete_diffusion(
        G_discrete_denoiser,
        torch.full((N_SAMPLES, G_discretized_gt_distribution.n_dims), MASK_TOKEN),
        G_alpha_schedule,
        num_steps=1,  # TODO
    )


def get_sampled_density_fn(samples: torch.Tensor):
    samples_np = samples.cpu().numpy()
    n_bins = G_discretized_gt_distribution.n_bins
    bin_edges = np.arange(n_bins + 1)
    hist, _, _ = np.histogram2d(
        samples_np[:, 0],
        samples_np[:, 1],
        bins=[bin_edges, bin_edges],
    )
    sampled_density = hist / N_SAMPLES

    def ret_fn(x):
        x_tensor = torch.as_tensor(x, dtype=torch.float32)
        bin_indices = G_discretized_gt_distribution._quantize(x_tensor)
        idx_1 = bin_indices[..., 0].numpy()
        idx_2 = bin_indices[..., 1].numpy()
        idx_1 = np.clip(idx_1, 0, n_bins - 1)
        idx_2 = np.clip(idx_2, 0, n_bins - 1)
        return sampled_density[idx_1, idx_2]

    return ret_fn


fig, axs = plt.subplots(1, 3, figsize=(18, 5))

ax_disc = axs[0]
ax_data = axs[1]
ax_samples = axs[2]

viz_img(
    lambda x: G_discretized_gt_distribution.log_prob_continuous(x).exp(),
    ax_disc,
    cax_label="probability",
)
G_discretized_gt_distribution.viz_support(ax_disc)
ax_disc.set_title(
    f"Discretized $p_X(x)$ ({G_discretized_gt_distribution.n_bins} bins/dim)"
)
ax_disc.set_xlabel(r"$x_1$")
ax_disc.set_ylabel(r"$x_2$")

viz_img(get_sampled_density_fn(train_x_disc), ax_data, cax_label="probability")
G_discretized_gt_distribution.viz_support(ax_data)
ax_data.set_title(
    f"Train data ({G_discretized_gt_distribution.n_bins}x{G_discretized_gt_distribution.n_bins} Grid)"
)
ax_data.set_xlabel(r"$x_1$")
ax_data.set_ylabel(r"$x_2$")


viz_img(get_sampled_density_fn(samples), ax_samples, cax_label="probability")
G_discretized_gt_distribution.viz_support(ax_samples)
ax_samples.set_title(
    f"Model samples ({G_discretized_gt_distribution.n_bins}x{G_discretized_gt_distribution.n_bins} Grid)"
)
ax_samples.set_xlabel(r"$x_1$")
ax_samples.set_ylabel(r"$x_2$")

for ax in axs:
    ax.set_xlim(G_x1_min_display, G_x1_max_display)
    ax.set_ylim(G_x2_min_display, G_x2_max_display)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## Guidance

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Categorical
from typing import Callable


def get_intermediate_potential( # TODO: play around
    discrete_denoiser: nn.Module,
    reward_net: nn.Module,
    xt: torch.Tensor,
    t: float,
    mask_token: int,
    guidance_scale: float,
) -> torch.Tensor:
    K = xt.shape[0]
    device = xt.device

    t_tensor = torch.full((K, 1), t, dtype=torch.float32, device=device)
    logits = discrete_denoiser((xt, t_tensor))

    x0_pred = logits.argmax(dim=-1)
    imagined_clean = torch.where(xt == mask_token, x0_pred, xt)
    imagined_clean = G_discretized_gt_distribution.to_continuous(imagined_clean)
    reward_logits = reward_net(imagined_clean).squeeze(-1)

    return torch.exp(guidance_scale * reward_logits)


@torch.no_grad()
def sample_discrete_diffusion_smc(
    discrete_denoiser: nn.Module,
    reward_net: nn.Module,
    num_particles: int,
    guidance_scale: float,
    num_steps: int,
    N_samples: int,
    alpha_schedule: Callable[[torch.Tensor], torch.Tensor],
) -> tuple[torch.Tensor, torch.Tensor]:
    device = next(discrete_denoiser.parameters()).device
    mask_token = discrete_denoiser.vocab_size
    seq_len = discrete_denoiser.seq_len
    N = num_particles

    xt = torch.full((N, seq_len), mask_token, dtype=torch.long, device=device)
    weights = torch.full((N,), 1.0 / N, device=device)

    ts = torch.linspace(1.0, 0.0, num_steps + 1, device=device)
    alphas = alpha_schedule(ts)

    g_curr = get_intermediate_potential(
        discrete_denoiser, reward_net, xt, ts[0].item(), mask_token, guidance_scale
    )

    for i in range(num_steps):
        t_curr = ts[i].item()
        t_next = ts[i + 1].item()
        alpha_curr = alphas[i].item()
        alpha_next = alphas[i + 1].item()

        t_tensor = torch.full((N, 1), t_curr, dtype=torch.float32, device=device)
        logits = discrete_denoiser((xt, t_tensor))
        probs = torch.softmax(logits, dim=-1)
        x0_pred = Categorical(probs).sample()

        p_unmask = (alpha_next - alpha_curr) / (1.0 - alpha_curr + 1e-9)
        p_unmask = max(0.0, min(1.0, p_unmask))

        is_masked = xt == mask_token
        unmask_trigger = torch.bernoulli(
            torch.full_like(xt, p_unmask, dtype=torch.float32)
        ).bool()
        should_reveal = is_masked & unmask_trigger

        xt_next = torch.where(should_reveal, x0_pred, xt)

        g_next = get_intermediate_potential(
            discrete_denoiser, reward_net, xt_next, t_next, mask_token, guidance_scale
        )

        weight_multiplier = g_next / (g_curr + 1e-9)
        weights = weights * weight_multiplier

        weights = weights / (weights.sum() + 1e-9)

        ess = 1.0 / (torch.sum(weights**2) + 1e-9)
        if ess < (N / 2.0):
            ancestors = torch.multinomial(weights, num_samples=N, replacement=True)
            xt_next = xt_next[ancestors]
            g_next = g_next[ancestors]
            weights = torch.full((N,), 1.0 / N, device=device)

        xt = xt_next
        g_curr = g_next

    ret_idx = Categorical(weights).sample((N_samples,))
    return xt[ret_idx]

In [ ]:
for p in G_reward_net.parameters():
    p.requires_grad_(False)
G_reward_net.eval()
G_discrete_denoiser.eval()

N_samples = 2048
configurations = [  # TODO
    {
        "num_particles": 100,
        "num_steps": 100,
        "guidance_scale": 0.1,
    },
]

fig, axs = plt.subplots(
    1, len(configurations) + 1, figsize=((len(configurations) + 1) * 6, 5)
)
metrics = G_sampled_distribution_to_optimize.compute_metrics(
    G_sampled_distribution_to_optimize.unlabeled_x
)
metrics["model"] = "initial_data"
records = [metrics]

viz_img(get_sampled_density_fn(train_x_disc), axs[0], cax_label="probability")
G_discretized_gt_distribution.viz_support(axs[0])

axs[0].set_xlabel(r"$x_1$")
axs[0].set_ylabel(r"$x_2$")


for ax, conf in zip(axs[1:], configurations):
    with torch.no_grad():
        samples = sample_discrete_diffusion_smc(
            discrete_denoiser=G_discrete_denoiser,
            reward_net=G_reward_net,
            N_samples=N_samples,
            alpha_schedule=G_alpha_schedule,
            **conf,
        )

    viz_img(
        get_sampled_density_fn(samples),
        ax,
        cax_label="probability",
    )
    G_discretized_gt_distribution.viz_support(ax)

    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")

    model_name = f"Conf({conf})"
    metrics = G_sampled_distribution_to_optimize.compute_metrics(
        G_discretized_gt_distribution.to_continuous(samples)
    )
    metrics["model"] = model_name
    records.append(metrics)

pd.DataFrame.from_records(records, index="model")